# A PINNs-XGBoost integrated framework for offshore monopile deformation prediction

**Paper:** Li, B., Qi, W., Wang, S., Song, Q., Gao, F. (2026). *A PINNs-XGBoost integrated framework for offshore monopile deformation prediction.* Acta Mechanica Sinica, 42, 451073. https://doi.org/10.1007/s10409-026-51073-x

**Carpeta origen:** `PINNs/1. mecanica de fluidos/A PINNs-XGBoost integrated framework for offshore monopile deformation prediction.pdf`

## Como se usan las PINNs en este paper

El paper modela la respuesta lateral de un monopilote offshore (viga de Euler-Bernoulli sobre cimentacion tipo Winkler no lineal) resolviendo la EDO de 4to orden:

$$EI\,\frac{d^4y}{dx^4} + p(x,y) = 0$$

con una red neuronal totalmente conectada $y^{NN}(\bar{x})$ ($\bar{x}=x/L_p$, activacion $\tanh$, 18 neuronas por capa oculta &mdash; valor optimo hallado por los autores en su analisis de sensibilidad, Fig. 5-6 del paper). La red se entrena minimizando una funcion de perdida compuesta por:

- $\ell_{PDE}$: residuo de la ecuacion de la viga (Eq. 9).
- $\ell_{top1}, \ell_{top2}$: condiciones de contorno en la cabeza del pilote (cortante y momento aplicados, Tabla 1).
- $\ell_{bot3}, \ell_{bot4}$: condiciones de contorno en la base (resortes de cortante $k_s$ y de momento $k_m$, Tabla 1).

Los pesos $\omega_1$-$\omega_4$ de cada termino se actualizan durante el entrenamiento con un algoritmo de balanceo adaptativo basado en la escala del gradiente (Eq. 10-12), y la tasa de aprendizaje seigue un *step decay* (Eq. 7).

En el paper completo, la relacion suelo-pilote no lineal $p(x,y)$ se obtiene con un modelo **XGBoost** entrenado sobre una base de datos propietaria de 221 curvas p-y (2554 puntos, 19 estudios), ajustada con un polinomio de 5to orden y reinyectada en $\ell_{PDE}$. Esa base de datos no es publica, por lo que en la Seccion 4.1 del propio paper (analisis de sensibilidad de arquitectura, Fig. 5-9) los autores validan la PINN aislada usando una relacion suelo explicita simplificada:

$$p(x,y) = m\,x\,\tanh(y) \qquad (Eq.\ 19)$$

con resortes lineales en la base: $F_b = k_s\,y_b$, $M_b = k_m\,\theta_b$ (Eq. 20-21, con $k_s=k_m=5\times10^{6}$ en el caso de referencia).

Este cuaderno reproduce fielmente **ese componente PINN validable de forma autocontenida** (arquitectura, funcion de perdida, condiciones de contorno, ponderacion adaptativa y *step-decay*), usando el caso de ejemplo de la Fig. 9(a) del paper (D = 1 m, $L_p$ = 60 m, $L_e$ = 8D, E = 210 GPa, I = 0.0079 m$^4$, m = 100000 N/m$^3$). El componente XGBoost (sustituible por cualquier regresor de curvas p-y) se documenta pero no se reentrena, al depender de datos no publicados.

## Repositorio publico de referencia

El PDF del paper **no incluye** un enlace a repositorio de codigo (no hay seccion "Data/Code availability" ni enlaces en las referencias). Tras buscar en GitHub un repositorio que implemente el mismo tipo de problema (PINN para la deflexion de una viga bajo un modelo de cimentacion tipo Winkler), el repositorio publico mas cercano encontrado es:

- **pollinico/PINN_beam_JAX** &mdash; https://github.com/pollinico/PINN_beam_JAX — PINN para aproximar la forma deformada de una viga (JAX), mismo tipo de EDO de 4to orden (Euler-Bernoulli) que en este paper, aunque sin el componente XGBoost ni las condiciones de contorno especificas de pilotes offshore.

Este cuaderno usa PyTorch (como el paper original, que reporta PyTorch 2.1.1) en lugar de JAX, pero replica el mismo principio de PINN para vigas que dicho repositorio.

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Parametros del problema (caso Fig. 9(a) del paper)

In [ ]:
# Parametros fisicos (pilote flexible, Fig. 9(a) del paper)
E = 210e9          # modulo elastico del acero [Pa]
I = 0.0079         # momento de inercia de la seccion [m^4]
EI = E * I         # rigidez a flexion [N m^2]
Lp = 60.0          # longitud embebida del pilote [m]
D = 1.0            # diametro del pilote [m]
Le = 8 * D         # excentricidad de la carga lateral [m]
m_soil = 100000.0  # rigidez del muelle de suelo, Eq. (19) [N/m^3]
ks = 5_000_000.0   # rigidez del resorte de cortante en la base, Eq. (20)
km = 5_000_000.0   # rigidez del resorte de momento en la base, Eq. (21)
P_list = [50e3, 100e3, 200e3, 300e3, 400e3]  # cargas laterales en cabeza [N]

N = 100  # numero de puntos de muestreo a lo largo del pilote, como en el paper
# .view() crea un tensor no-hoja; requires_grad_() se aplica despues para que
# x_bar sea una hoja real y sus gradientes se puedan resetear entre epocas.
x_bar = torch.linspace(0.0, 1.0, N, device=device).view(-1, 1).requires_grad_(True)

## 2. Red neuronal (Eq. 3-6)

Red totalmente conectada con activacion $\tanh$. El paper reporta que 18 neuronas por capa oculta ofrece el mejor balance precision/estabilidad (Fig. 5-6).

In [ ]:
class PINN(nn.Module):
    def __init__(self, n_hidden_layers=4, n_neurons=18):
        super().__init__()
        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden_layers - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def nth_derivative(y, x, n):
    """d^n y / d x_bar^n via autograd, retiene el grafo para derivadas de orden superior."""
    for _ in range(n):
        grads = torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y),
                                     create_graph=True, retain_graph=True)[0]
        y = grads
    return y

## 3. Funcion de perdida (Eq. 8-9, Tabla 1)

$$\ell = \ell_{PDE} + \omega_1\ell_{top1} + \omega_2\ell_{top2} + \omega_3\ell_{bot3} + \omega_4\ell_{bot4}$$

Como $\bar{x}=x/L_p$, las derivadas fisicas se obtienen de las derivadas respecto a $\bar{x}$ escalando por $1/L_p^n$ (regla de la cadena).

In [ ]:
def soil_reaction(x_phys, y):
    """p(x, y) = m * x * tanh(y), Eq. (19) del paper."""
    return m_soil * x_phys * torch.tanh(y)


def compute_losses(model, x_bar, P):
    y = model(x_bar)
    x_phys = x_bar * Lp

    dy = nth_derivative(y, x_bar, 1) / Lp
    d2y = nth_derivative(y, x_bar, 2) / Lp**2
    d3y = nth_derivative(y, x_bar, 3) / Lp**3
    d4y = nth_derivative(y, x_bar, 4) / Lp**4

    # l_PDE: residuo de la ecuacion de la viga, Eq. (9)
    p = soil_reaction(x_phys, y)
    residual = EI * d4y + p
    l_pde = torch.mean(residual**2)

    # Condiciones de contorno, Tabla 1 (x0 = cabeza del pilote, x1 = base)
    y0, dy0, d2y0, d3y0 = y[0], dy[0], d2y[0], d3y[0]
    y1, dy1, d2y1, d3y1 = y[-1], dy[-1], d2y[-1], d3y[-1]

    l_top1 = (EI * d3y0 - P)**2                      # cortante en cabeza = P
    l_top2 = (EI * d2y0 - P * Le)**2                  # momento en cabeza = P*Le
    l_bot3 = (EI * d3y1 + ks * y1)**2                 # resorte de cortante en base
    l_bot4 = (EI * d2y1 - km * dy1)**2                # resorte de momento en base

    return l_pde, l_top1.squeeze(), l_top2.squeeze(), l_bot3.squeeze(), l_bot4.squeeze()

## 4. Entrenamiento: pesos adaptativos (Eq. 10-12) + step-decay LR (Eq. 7)

In [ ]:
def train_pinn(P, epochs=8000, eta_max=1e-3, decay_period=1000, beta=0.9):
    model = PINN(n_hidden_layers=4, n_neurons=18).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=eta_max)
    weights = torch.ones(4, device=device)

    for epoch in range(epochs):
        # Eq. (7): step decay learning rate
        lr = eta_max * (0.5 ** (epoch // decay_period))
        for g in optimizer.param_groups:
            g['lr'] = lr

        # Eq. (10)-(12): actualizacion adaptativa de pesos cada 10 epocas.
        # Se usa un forward pass separado (grafo propio) para no interferir
        # con el grafo del paso de optimizacion principal de mas abajo.
        if epoch % 10 == 0:
            l_pde_w, l_top1_w, l_top2_w, l_bot3_w, l_bot4_w = compute_losses(model, x_bar, P)
            bc_losses_w = [l_top1_w, l_top2_w, l_bot3_w, l_bot4_w]
            grad_pde = torch.autograd.grad(l_pde_w, model.parameters(),
                                            retain_graph=True, allow_unused=True)
            max_g_pde = max(g.abs().max() for g in grad_pde if g is not None)
            for i, li in enumerate(bc_losses_w):
                is_last = (i == len(bc_losses_w) - 1)
                grads_i = torch.autograd.grad(li, model.parameters(),
                                               retain_graph=not is_last, allow_unused=True)
                g_i = torch.sqrt(sum((g**2).sum() for g in grads_i if g is not None) + 1e-12)
                bar_w = max_g_pde / (g_i + 1e-12)
                weights[i] = beta * weights[i] + (1 - beta) * bar_w

        optimizer.zero_grad()
        l_pde, l_top1, l_top2, l_bot3, l_bot4 = compute_losses(model, x_bar, P)
        bc_losses = torch.stack([l_top1, l_top2, l_bot3, l_bot4])
        loss = l_pde + torch.sum(weights * bc_losses)
        loss.backward()
        optimizer.step()
        if x_bar.grad is not None:
            x_bar.grad.zero_()

        if epoch % 2000 == 0:
            print(f'P={P/1e3:.0f} kN | epoch {epoch:5d} | lr={lr:.2e} | loss={loss.item():.4e}')

    return model


trained_models = {}
for P in P_list:
    trained_models[P] = train_pinn(P, epochs=6000)

## 5. Resultados: curva de deflexion y(x) (comparable a Fig. 9(a) del paper)

In [ ]:
plt.figure(figsize=(6, 7))
x_plot = np.linspace(0, Lp, N)
for P, model in trained_models.items():
    with torch.no_grad():
        y_pred = model(x_bar).cpu().numpy().flatten()
    plt.plot(y_pred, -x_plot, label=f'P = {P/1e3:.0f} kN')

plt.xlabel('y (m) — deflexion lateral')
plt.ylabel('profundidad z (m) — 0 en cabeza, -60 en punta')
plt.title('Deflexion del pilote via PINN\n(reproduccion del caso Fig. 9(a) del paper)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Nota sobre el componente XGBoost

En el framework completo del paper, `soil_reaction(x, y)` (celda 3) no es la formula analitica `Eq. 19` sino un modelo **XGBoost** entrenado sobre 221 curvas p-y experimentales (2554 puntos, 19 estudios recopilados en Li et al., no publicados) que predice `p` para un `y` dado, seguido de un ajuste polinomico de 5to orden que se reevalua en cada epoca de entrenamiento de la PINN (ver Eq. 13-14, Fig. 1, Fig. 3 y Tabla 2 del paper). Para reproducir esa parte con datos propios bastaria con:

```python
from xgboost import XGBRegressor
xgb_model = XGBRegressor(n_estimators=716, max_depth=7, learning_rate=0.0505,
                          subsample=0.728, min_child_weight=4.2973)  # hiperparametros optimos, Tabla 2
xgb_model.fit(X_train, p_train)  # X: (Dr, phi_cr, sigma'_v, Lp/D, z/D, y/D, gamma'/gamma)
# despues: ajustar p-y con np.polyfit(y_vals, xgb_model.predict(...), deg=5) por profundidad
```

y sustituir `soil_reaction` por la evaluacion de ese polinomio dentro de `compute_losses`. El resto del pipeline (arquitectura de la PINN, condiciones de contorno, pesos adaptativos, step-decay) es exactamente el implementado arriba.